In [ ]:
!pip install requests pandas python-dateutil tqdm pyarrow

In [1]:
import requests
import json
import pandas as pd
import time
from datetime import datetime
from tqdm import tqdm
from dateutil.relativedelta import relativedelta

# ===============================================================
# CONFIG / DATA
# ===============================================================
API_KEY = "77eb6bb3-185d-4f06-9ef2-905ede37414a"
BASE_URL = "https://tools.kinds.or.kr/search/news"

# KOSPI company metadata (same as your list)
kospi_data = [
    {"Company": "삼성전자", "Ticker": "005930.KS", "Type": "Tech", "CEO": "이재용(회장), 한종희, 전영현", "Sector": "Semiconductor", "Alias": ["삼성", "삼전"]},
    {"Company": "삼성물산", "Ticker": "028260.KS", "Type": "Non-Tech", "CEO": "이재용(실질총수), 오세철, 이재언", "Sector": "Construction/Trading", "Alias": ["물산"]},
    {"Company": "삼성생명", "Ticker": "032830.KS", "Type": "Non-Tech", "CEO": "이재용(대주주), 홍원학", "Sector": "Finance", "Alias": []},
    {"Company": "삼성화재", "Ticker": "000810.KS", "Type": "Non-Tech", "CEO": "이재용(대주주), 이문화", "Sector": "Finance", "Alias": []},
    {"Company": "삼성SDI", "Ticker": "006400.KS", "Type": "Tech", "CEO": "이재용(총수), 최윤호", "Sector": "Batteries", "Alias": ["SDI"]},
    {"Company": "삼성바이오로직스", "Ticker": "207940.KS", "Type": "Tech", "CEO": "이재용(총수), 존림", "Sector": "Bio", "Alias": ["삼바"]},
    {"Company": "삼성중공업", "Ticker": "010140.KS", "Type": "Non-Tech", "CEO": "이재용(총수), 최성안", "Sector": "Shipbuilding", "Alias": []},

    {"Company": "SK하이닉스", "Ticker": "000660.KS", "Type": "Tech", "CEO": "최태원(회장), 곽노정", "Sector": "Semiconductor", "Alias": ["하이닉스"]},
    {"Company": "SK", "Ticker": "034730.KS", "Type": "Non-Tech", "CEO": "최태원(회장), 장용호", "Sector": "Holding", "Alias": []},
    {"Company": "SK이노베이션", "Ticker": "096770.KS", "Type": "Non-Tech", "CEO": "최태원(회장), 박상규", "Sector": "Energy", "Alias": ["SK이노"]},
    {"Company": "SK텔레콤", "Ticker": "017670.KS", "Type": "Tech", "CEO": "최태원(회장), 유영상", "Sector": "Telecom", "Alias": ["SKT"]},

    {"Company": "현대차", "Ticker": "005380.KS", "Type": "Non-Tech", "CEO": "정의선(회장), 장재훈", "Sector": "Auto", "Alias": ["현차", "현대"]},
    {"Company": "기아", "Ticker": "000270.KS", "Type": "Non-Tech", "CEO": "정의선(회장), 송호성", "Sector": "Auto", "Alias": []},
    {"Company": "현대모비스", "Ticker": "012330.KS", "Type": "Non-Tech", "CEO": "정의선(회장), 이규석", "Sector": "Auto Parts", "Alias": ["모비스"]},

    {"Company": "LG", "Ticker": "003550.KS", "Type": "Non-Tech", "CEO": "구광모(회장)", "Sector": "Holding", "Alias": []},
    {"Company": "LG전자", "Ticker": "066570.KS", "Type": "Tech", "CEO": "구광모(회장), 조주완", "Sector": "Electronics", "Alias": ["LG전", "엘지전자"]},
    {"Company": "LG에너지솔루션", "Ticker": "373220.KS", "Type": "Tech", "CEO": "구광모(회장), 김동명", "Sector": "Batteries", "Alias": ["LG엔솔", "엔솔"]},
    {"Company": "LG화학", "Ticker": "051910.KS", "Type": "Non-Tech", "CEO": "구광모(회장), 신학철", "Sector": "Chemicals", "Alias": ["LG화"]},
    {"Company": "LG생활건강", "Ticker": "051900.KS", "Type": "Non-Tech", "CEO": "구광모(회장), 이정애", "Sector": "Cosmetics", "Alias": ["엘지생건"]},

    {"Company": "롯데케미칼", "Ticker": "011170.KS", "Type": "Non-Tech", "CEO": "신동빈(회장), 이훈기", "Sector": "Chemicals", "Alias": []},
    {"Company": "롯데쇼핑", "Ticker": "023530.KS", "Type": "Non-Tech", "CEO": "신동빈(회장), 김상현", "Sector": "Retail", "Alias": []},

    {"Company": "한화솔루션", "Ticker": "009830.KS", "Type": "Non-Tech", "CEO": "김승연(회장), 김동관(부회장)", "Sector": "Chemicals/Solar", "Alias": ["한화솔"]},
    {"Company": "GS", "Ticker": "078930.KS", "Type": "Non-Tech", "CEO": "허태수(회장)", "Sector": "Holding/Energy", "Alias": []},
    {"Company": "HD현대", "Ticker": "267250.KS", "Type": "Non-Tech", "CEO": "권오갑(회장), 정기선(부회장)", "Sector": "Holding", "Alias": []},
    {"Company": "HD현대중공업", "Ticker": "329180.KS", "Type": "Non-Tech", "CEO": "정기선(부회장), 이상균", "Sector": "Shipbuilding", "Alias": ["현대중공업", "HD현중"]},

    {"Company": "CJ제일제당", "Ticker": "097950.KS", "Type": "Non-Tech", "CEO": "이재현(회장), 강신호", "Sector": "Food", "Alias": ["제일제당"]},
    {"Company": "두산에너빌리티", "Ticker": "034020.KS", "Type": "Non-Tech", "CEO": "박지원(회장)", "Sector": "Heavy Ind", "Alias": []},
    {"Company": "대한항공", "Ticker": "003490.KS", "Type": "Non-Tech", "CEO": "조원태(회장)", "Sector": "Transport", "Alias": []},

    {"Company": "네이버", "Ticker": "035420.KS", "Type": "Tech", "CEO": "이해진(GIO), 최수연", "Sector": "Platform", "Alias": ["NAVER"]},
    {"Company": "카카오", "Ticker": "035720.KS", "Type": "Tech", "CEO": "김범수(창업자), 정신아", "Sector": "Platform", "Alias": ["KAKAO"]},
    {"Company": "셀트리온", "Ticker": "068270.KS", "Type": "Tech", "CEO": "서정진(회장), 김형기", "Sector": "Bio", "Alias": ["셀트"]},
    {"Company": "엔씨소프트", "Ticker": "036570.KS", "Type": "Tech", "CEO": "김택진(대표/오너), 박병무", "Sector": "Game", "Alias": ["엔씨"]},
    {"Company": "넷마블", "Ticker": "251270.KS", "Type": "Tech", "CEO": "방준혁(의장), 권영식", "Sector": "Game", "Alias": []},
    {"Company": "크래프톤", "Ticker": "259960.KS", "Type": "Tech", "CEO": "장병규(의장), 김창한", "Sector": "Game", "Alias": []},
    {"Company": "하이브", "Ticker": "352820.KS", "Type": "Non-Tech", "CEO": "방시혁(의장), 이재상", "Sector": "Ent", "Alias": []},

    {"Company": "에코프로", "Ticker": "086520.KQ", "Type": "Tech", "CEO": "이동채(창업주), 송호준", "Sector": "Batteries", "Alias": []},
    {"Company": "금양", "Ticker": "001570.KS", "Type": "Tech", "CEO": "류광지(회장/오너)", "Sector": "Batteries", "Alias": []},
    {"Company": "DB손해보험", "Ticker": "005830.KS", "Type": "Non-Tech", "CEO": "김남호(회장), 정종표", "Sector": "Finance", "Alias": []},
    {"Company": "아모레퍼시픽", "Ticker": "090430.KS", "Type": "Non-Tech", "CEO": "서경배(회장), 김승환", "Sector": "Cosmetics", "Alias": ["아모레"]},

    {"Company": "POSCO홀딩스", "Ticker": "005490.KS", "Type": "Non-Tech", "CEO": "장인화(회장)", "Sector": "Steel", "Alias": ["포스코"]},
    {"Company": "포스코퓨처엠", "Ticker": "003670.KS", "Type": "Tech", "CEO": "유병옥(사장)", "Sector": "Materials", "Alias": ["퓨처엠"]},
    {"Company": "KT", "Ticker": "030200.KS", "Type": "Tech", "CEO": "김영섭(대표)", "Sector": "Telecom", "Alias": []},
    {"Company": "KT&G", "Ticker": "033780.KS", "Type": "Non-Tech", "CEO": "방경만(사장)", "Sector": "Tobacco", "Alias": ["케이티앤지"]},

    {"Company": "HMM", "Ticker": "011200.KS", "Type": "Non-Tech", "CEO": "김경배(사장)", "Sector": "Logistics", "Alias": []},
    {"Company": "S-Oil", "Ticker": "010950.KS", "Type": "Non-Tech", "CEO": "안와르 알 히즈아지", "Sector": "Energy", "Alias": ["에쓰오일"]},
    {"Company": "한국전력", "Ticker": "015760.KS", "Type": "Non-Tech", "CEO": "김동철", "Sector": "Utilities", "Alias": ["한전"]},

    {"Company": "KB금융", "Ticker": "105560.KS", "Type": "Non-Tech", "CEO": "양종희(회장)", "Sector": "Finance", "Alias": ["KB"]},
    {"Company": "신한지주", "Ticker": "055550.KS", "Type": "Non-Tech", "CEO": "진옥동(회장)", "Sector": "Finance", "Alias": ["신한"]},
    {"Company": "하나금융지주", "Ticker": "086790.KS", "Type": "Non-Tech", "CEO": "함영주(회장)", "Sector": "Finance", "Alias": ["하나금융"]},
    {"Company": "기업은행", "Ticker": "024110.KS", "Type": "Non-Tech", "CEO": "김성태(은행장)", "Sector": "Finance", "Alias": ["IBK"]},
]

df_info = pd.DataFrame(kospi_data)

# Alias mapping
ALIAS_MAP = {r["Company"]: r["Alias"] for r in kospi_data}


# ===============================================================
# HELPER FUNCTIONS
# ===============================================================
def split_ceo_name(ceo_name: str):
    """
    Extract clean Korean CEO name parts (without titles) for search and classification.
    """
    if not ceo_name:
        return []
    cleaned = ceo_name.replace("-", " ").replace(",", " ")
    parts = cleaned.split()

    result = []
    # Simplified list of common titles/roles to exclude when extracting names
    titles_to_exclude = ["회장", "부회장", "사장", "대표", "총수", "오너", "GIO", "창업주", "의장", "은행장", "실질총수", "대주주", "대표/오너"]
    
    for p in parts:
        # Check if the part contains Korean characters, is not just a title/role
        if any('가' <= ch <= '힣' for ch in p) and not any(title in p for title in titles_to_exclude):
            name = p.split("(")[0].strip()
            result.append(name)
        # Handle the non-Korean name explicitly
        elif '알 히즈아지' in p: 
            result.append('히즈아지')

    return list(dict.fromkeys(result))


def build_query(company, alias_map=ALIAS_MAP, ceo_name=""):
    """
    Build OR keyword search string for KINDS API.
    Example: 삼성전자 OR 삼전 OR 이재용 OR 한종희 ...
    """
    keywords = [company] + alias_map.get(company, [])
    
    # NOTE: The CEO names are split here to ensure individual names are searched
    ceo_parts = split_ceo_name(ceo_name) 
    keywords += ceo_parts

    return " OR ".join([k for k in keywords if k])


def fetch_all_news_for_company(company, query_str, full_ceo_name, company_type, api_key=API_KEY, max_articles_per_window=2000):
    """
    Fetch news from KINDS API in monthly windows from 2023-01-01 until today.
    Adds CEO metadata and classification columns.
    """
    all_docs = []
    start_date = datetime(2023, 1, 1)
    end_date = datetime.now()
    
    # Get the clean CEO names for news type classification
    leadership_names = split_ceo_name(full_ceo_name)

    while start_date <= end_date:
        window_end = (start_date + relativedelta(months=1)) - relativedelta(days=1)
        if window_end > end_date:
            window_end = end_date

        start_index = 0
        batch_size = 1000

        while True:
            payload = {
                "access_key": api_key,
                "argument": {
                    "query": query_str,
                    "published_at": {
                        "from": start_date.strftime("%Y-%m-%d"),
                        "until": window_end.strftime("%Y-%m-%d")
                    },
                    "sort": {"date": "desc"},
                    "hilight": 200, 
                    "return_size": batch_size,
                    "start_index": start_index,
                    "fields": [
                        "news_id", "title", "provider", "published_at",
                        "category", "content", "hilight"
                    ]
                }
            }

            headers = {"Content-Type": "application/json; charset=UTF-8"}

            try:
                resp = requests.post(BASE_URL, headers=headers,
                                     data=json.dumps(payload), timeout=30)

                if resp.status_code != 200:
                    print(f"[{company}] HTTP {resp.status_code} - stopping window.")
                    break

                data = resp.json()
                docs = data.get("return_object", {}).get("documents", [])
                if not docs:
                    break

                for d in docs:
                    d["target_company"] = company
                    d["query"] = query_str
                    d["window_start"] = start_date.strftime("%Y-%m-%d")
                    d["window_end"] = window_end.strftime("%Y-%m-%d")
                    
                    # ----------------------------------------------------
                    # NEW LOGIC: Classify News Type & Identify Mentioned Names
                    # ----------------------------------------------------
                    news_text = d.get("hilight", "") + " " + d.get("title", "")
                    mentioned_names = []

                    if leadership_names:
                        for name in leadership_names:
                            # Check for the name in the title/hilight text
                            if name in news_text:
                                mentioned_names.append(name)
                    
                    # Store the full CEO string (as requested)
                    d["CEO_Leadership_Name"] = full_ceo_name
                    
                    # New Column: Mentioned Leadership Name(s)
                    d["Mentioned_Leadership_Name"] = ", ".join(mentioned_names) if mentioned_names else None
                    
                    # News Type Classification
                    if mentioned_names:
                        d["News_Type"] = "Leadership News"
                    else:
                        d["News_Type"] = "Company News"
                    # ----------------------------------------------------
                    
                    # Already collected metadata
                    d["Company_Type"] = company_type


                all_docs.extend(docs)

                # Stop when finished
                if len(docs) < batch_size or len(all_docs) >= max_articles_per_window:
                    break

                start_index += batch_size
                time.sleep(0.2)

            except Exception as e:
                print(f"[{company}] Error: {e}")
                break

        start_date += relativedelta(months=1)

    return pd.DataFrame(all_docs) if all_docs else pd.DataFrame()


# ===============================================================
# MAIN EXECUTION (DATA ACQUISITION ONLY)
# ===============================================================
if __name__ == "__main__":
    print(f"Starting raw KINDS data collection for {len(df_info)} companies.\n")

    all_frames = []

    for company in tqdm(df_info["Company"].tolist(), desc="Companies"):
        meta = df_info[df_info["Company"] == company].iloc[0]
        
        ceo_name = meta.get("CEO", "")
        company_type = meta.get("Type", "Non-Tech") # Tech/Non-Tech Column
        
        # NOTE: build_query uses split_ceo_name, so it searches for individual names.
        query = build_query(company, ALIAS_MAP, ceo_name)

        df_company = fetch_all_news_for_company(company, query, ceo_name, company_type, API_KEY)
        if df_company.empty:
            continue

        # Add remaining metadata
        df_company["company"] = company
        df_company["ticker"] = meta.get("Ticker", "")
        df_company["sector"] = meta.get("Sector", "")
        # CEO_Leadership_Name, Mentioned_Leadership_Name, News_Type, and Company_Type 
        # are already added inside fetch_all_news_for_company

        all_frames.append(df_company)

    if not all_frames:
        print("No articles collected.")
        exit()

    final_df = pd.concat(all_frames, ignore_index=True)
    
    # Drop columns not needed in the final output but used during fetch
    if "content" in final_df.columns:
        final_df.drop(columns=["content", "hilight"], errors='ignore', inplace=True)
        
    final_df.drop_duplicates(subset=["news_id", "title"], keep="first", inplace=True)

    out_name = f"KOSPI_RawNews_Classified_v2_{datetime.now().strftime('%Y%m%d')}.parquet"
    final_df.to_parquet(out_name, index=False)

    print("====================================")
    print("RAW DATA ACQUISITION COMPLETED")
    print("Saved:", out_name)
    print("Total articles collected:", len(final_df))
    print("====================================")
    
    # Display the newly requested columns
    print("Sample Output (New Columns Highlighted):")
    print(final_df[["company", "published_at", "title", "Company_Type", "CEO_Leadership_Name", "Mentioned_Leadership_Name", "News_Type"]].head(10))

Starting raw KINDS data collection for 50 companies.



Companies:   0%|                                                                                                                     | 0/50 [00:00<?, ?it/s]


KeyboardInterrupt: 